# Generate wrapped interferograms and coherence maps using OPERA CSLC-S1

---    

This notebook:
- Searches OPERA CSLC-S1 products for your AOI + date range
- Subsets CSLCs **before download** using opera-utils
- Builds interferograms/coherence and merges bursts
- Exports mosaics and visualizations

**Quick start**
1) Set parameters in the next cell (AOI, date range, pairing)
2) Run cells top-to-bottom
3) Outputs land in `savedir/`

**Key toggles**
- `REPROJECT_FOR_DISPLAY`: reproject plots to WGS84 when True
- `SAVE_WGS84`: save WGS84 GeoTIFF mosaics when True
- `DOWNLOAD_WITH_PROGRESS`: show progress bar for downloads
- `USE_WATER_YEAR`: Oct–Sep calendar layout when True
- `pair_mode` / `t_span`: control IFG pairing (all vs fixed separation)

**Outputs**
- Subset CSLC H5: `savedir/subset_cslc/*.h5`
- Mosaics (IFG/COH, native CRS): `savedir/tifs/merged_ifg_*`, `merged_coh_*`
- WGS84 mosaics: `savedir/tifs/WGS84/merged_ifg_WGS84_*`, `merged_coh_WGS84_*`, `merged_amp_WGS84_*`
- Amplitude mosaics (native CRS): `savedir/tifs/merged_amp_*.tif`
- GIFs: `savedir/gifs/*.gif`




### Data Used in the Example:   

- **10 meter (Northing) x 5 meter (Easting) North America OPERA Coregistered Single Look Complex from Sentinel-1 products**
    - This dataset contains Level-2 OPERA coregistered single-look-complex (CSLC) data from Sentinel-1 (S1). <span style="color:red"> The data in this example are geocoded CSLC-S1 data covering Palos Verdes landslides, California, USA</span>. 
    
    - The OPERA project is generating geocoded burst-wise CSLC-S1 products over North America which includes USA and US Territories within 200 km from the US border, Canada, and all mainland countries from the southern US border down to and including Panama. Each pixel within a burst SLC is represented by a complex number and contains both the amplitude and phase information. The CSLC-S1 products are distributed over projected map coordinates using the Universal Transverse Mercator (UTM) projection with spacing in the X- and Y-directions of 5 m and 10 m, respectively. Each OPERA CSLC-S1 product is distributed as a HDF5 file following the CF-1.8 convention with separate groups containing the data raster layers, the low-resolution correction layers, and relevant product metadata.

    - For more information about the OPERA project and other products please visit our website at https://www.jpl.nasa.gov/go/opera .

Please refer to the [OPERA Product Specification Document](https://d2pn8kiwq2w21t.cloudfront.net/documents/OPERA_CSLC-S1_ProductSpec_v1.0.0_D-108278_Initial_2023-09-11_URS321269.pdf) for details about the CSLC-S1 product.

*Prepared by Al Handwerger and M. Grace Bato*

---

## 0. Setup your conda environment

Assuming you have conda installed. Open your terminal and run the following:
```

# Create the OPERA CSLC environment
conda env create -f environment_opera_cslc_landslides.yml
conda activate opera_cslc_landslides
python -m ipykernel install --user --name opera_cslc_landslides

```

---    

## 1. Load Python modules

In [ ]:
## Load necessary modules
%load_ext watermark

import asf_search as asf
import geopandas as gpd
import pandas as pd

import numpy as np
from netrc import netrc
from subprocess import Popen
from platform import system
from getpass import getpass
import folium
import datetime as dt
from shapely.geometry import box
from shapely.geometry import Point
import shapely.wkt as wkt
import rioxarray
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import cartopy.crs as ccrs
import xarray as xr
import rasterio
from rasterio.transform import from_origin
from rasterio import merge
from rasterio.crs import CRS

import os, sys
proj_dir = os.path.join(sys.prefix, "share", "proj")
os.environ["PROJ_LIB"] = proj_dir  # for older PROJ
os.environ["PROJ_DATA"] = proj_dir  # for newer PROJ


%watermark --iversions

import os

In [ ]:
# Environment check
import sys
import importlib

REQUIRED_PKGS = [
    'asf_search','cartopy','folium','geopandas','h5py','imageio','matplotlib','numpy','pandas',
    'pyproj','rasterio','rioxarray','shapely','xarray','opera_utils','tqdm','rich'
]
missing = []
for pkg in REQUIRED_PKGS:
    try:
        importlib.import_module(pkg)
    except Exception:
        missing.append(pkg)

if missing:
    raise ImportError("Missing packages: " + ', '.join(missing) + ". Activate opera_cslc env or install from environment_opera_cslc.yml")

print(f"Python: {sys.executable}")
print('Environment check OK')


In [ ]:
## Load plotting module
import matplotlib.pyplot as plt
%matplotlib inline
%config InlineBackend.figure_format='retina'

In [ ]:
## Load pandas and setup config to expand the display of the database
import pandas as pd
# pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [ ]:
def _maybe_reproject(da):
    if REPROJECT_FOR_DISPLAY:
        return da.rio.reproject("EPSG:4326")
    return da


In [ ]:
## Avoid lots of these warnings printing to notebook from asf_search
import warnings
warnings.filterwarnings('ignore')

## 2. Set up your NASA Earthdata Login Credentials

In [ ]:
urs = 'urs.earthdata.nasa.gov'
prompts = ['Enter NASA Earthdata Login Username: ',
           'Enter NASA Earthdata Login Password: ']

netrc_name = "_netrc" if system() == "Windows" else ".netrc"
netrc_path = os.path.expanduser(f"~/{netrc_name}")

def write_netrc():
    username = getpass(prompt=prompts[0])
    password = getpass(prompt=prompts[1])
    with open(netrc_path, 'a') as f:
        f.write(f"\nmachine {urs}\n")
        f.write(f"login {username}\n")
        f.write(f"password {password}\n")
    os.chmod(netrc_path, 0o600)

def has_urs_credentials():
    try:
        creds = netrc(netrc_path).authenticators(urs)
        return creds is not None
    except (FileNotFoundError, NetrcParseError):
        return False

if not has_urs_credentials():
    if not os.path.exists(netrc_path):
        open(netrc_path, 'w').close()
    write_netrc()

import os
os.environ["GDAL_HTTP_NETRC"] = "YES"
os.environ["GDAL_HTTP_NETRC_FILE"] = netrc_path


## 3. Enter user-defined parameters

In [ ]:
# User parameters (edit these)
# AOI is a WKT polygon in EPSG:4326
## Enter user-defined parameters
SITE_NAME = "Palos_Verdes_Landslides"  # used for output folder naming
aoi = "POLYGON((-118.3955 33.7342,-118.3464 33.7342,-118.3464 33.7616,-118.3955 33.7616,-118.3955 33.7342))"
orbitPass = "DESCENDING"
pathNumber = 71
# Optional burst selection before download
# Use subswath (e.g., 'IW2', 'IW3') or specific OPERA burst ID (e.g., 'T071_151230_IW3')
BURST_SUBSWATH = 'IW3'  # e.g., 'IW2' or ['IW2', 'IW3'] or None  
BURST_ID = None        # e.g., 'T071_151230_IW3' or list of burst IDs
dateStart = dt.datetime.fromisoformat('2017-10-01 00:00:00')         #'YYYY-MM-DD HH:MM:SS'
dateEnd = dt.datetime.fromisoformat('2017-12-01 23:59:59')           #'YYYY-MM-DD HH:MM:SS'

# Pairing options
pair_mode = 't_span'   # 'all' or 't_span'
pair_t_span_days = 12        # int or list of ints (e.g., [12, 24])

# Multilooking (spatial averaging)
# Set either MULTILOOK (looks_y, looks_x) OR TARGET_PIXEL_M (meters). TARGET overrides MULTILOOK.
MULTILOOK = (1, 1)  # e.g., (3, 6) for 30m from (dy=10m, dx=5m)
TARGET_PIXEL_M = None  # e.g., 30.0 or 90.0

REPROJECT_FOR_DISPLAY = False  # set True to reproject plots to EPSG:4326
SAVE_WGS84 = False  # set True to save WGS84 GeoTIFF mosaics



DOWNLOAD_WITH_PROGRESS = True  # set True for per-file progress bar

import os

min_t_span_days = 12         # minimum separation (days)
max_t_span_days = 12     # maximum separation (days) or None
max_pairs_per_burst = None  # int or None
max_pairs_total = None      # int or None

# Calendar settings
USE_WATER_YEAR = True  # True: Oct–Sep, False: Jan–Dec

DOWNLOAD_PROCESSES = min(8, max(2, (os.cpu_count() or 4) // 2))
# DOWNLOAD_BATCH_SIZE = 5


# Normalize name for filesystem (letters/numbers/_/- only)
import re
site_slug = re.sub(r"[^A-Za-z0-9_-]+", "", SITE_NAME)
orbit_code = orbitPass[0].upper()  # 'A' or 'D'
savedir = f'./{site_slug}_{orbit_code}{pathNumber:03d}/'

# Water mask options
# in params cell
WATER_MASK_PATH = f"{savedir}/water_mask/water_mask_esa_wc2021.tif"
APPLY_WATER_MASK = True


In [ ]:
# ESA WorldCover 2021 water mask (GDAL-only)
from pathlib import Path
import geopandas as gpd
import shapely.wkt
import rasterio
from osgeo import gdal

ESA_WC_GRID_URL = "https://esa-worldcover.s3.eu-central-1.amazonaws.com/esa_worldcover_grid.fgb"
ESA_WC_BASE_URL = "https://esa-worldcover.s3.eu-central-1.amazonaws.com/v200/2021/map"
# ESA WorldCover class codes: 80 = Permanent water bodies
ESA_WC_WATER_CLASSES = {80}


def build_worldcover_water_mask(aoi_wkt, out_path, target_res_deg=None):
    # Create a binary land mask from ESA WorldCover (1=land, 0=water).
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    if out_path.exists():
        return out_path


    aoi_geom = shapely.wkt.loads(aoi_wkt)
    # Load tile grid and select intersecting tiles
    grid = gpd.read_file(ESA_WC_GRID_URL)
    grid = grid.to_crs("EPSG:4326")
    # Find tile id column (varies by grid version)
    tile_col = next((c for c in grid.columns if 'tile' in c.lower()), None)
    if tile_col is None:
        raise RuntimeError(f"No tile column found in grid columns: {list(grid.columns)}")
    tiles = grid[grid.intersects(aoi_geom)][tile_col].tolist()
    if not tiles:
        raise RuntimeError("No WorldCover tiles intersect AOI")

    print(f"Selected tiles: {tiles}")

    tile_urls = [
        f"{ESA_WC_BASE_URL}/ESA_WorldCover_10m_2021_v200_{t}_Map.tif"
        for t in tiles
    ]

    # Quick URL check for first tile
    first_url = tile_urls[0]
    try:
        _ = gdal.Open(first_url)
    except Exception as e:
        raise RuntimeError(f"GDAL cannot open first tile URL: {first_url}\n{e}")

    vrt_path = out_path.with_suffix(".vrt")
    gdal.BuildVRT(str(vrt_path), tile_urls)

    minx, miny, maxx, maxy = aoi_geom.bounds
    warp_kwargs = dict(
        format="GTiff",
        outputBounds=[minx, miny, maxx, maxy],
        multithread=True,
    )
    if target_res_deg is not None:
        warp_kwargs.update(dict(xRes=target_res_deg, yRes=target_res_deg, targetAlignedPixels=True))

    tmp_map = out_path.with_name(out_path.stem + "_map.tif")
    warp_ds = gdal.Warp(str(tmp_map), str(vrt_path), **warp_kwargs)
    if warp_ds is None:
        raise RuntimeError("GDAL Warp returned None. Check network access/URL.")
    warp_ds = None

    with rasterio.open(tmp_map) as src:
        data = src.read(1)
        profile = src.profile

    mask = (~np.isin(data, list(ESA_WC_WATER_CLASSES))).astype("uint8")
    profile.update(dtype="uint8", count=1, nodata=0)

    with rasterio.open(out_path, "w", **profile) as dst:
        dst.write(mask, 1)

    return out_path

WATER_MASK_PATH = build_worldcover_water_mask(aoi, f"{savedir}/water_mask/water_mask_esa_wc2021.tif")
APPLY_WATER_MASK = True


## 4. Query OPERA CSLCs using `asf_search`

In [ ]:
## Search for OPERA CSLC data in ASF DAAC
try:
    search_params = dict(
        intersectsWith= aoi,
        dataset='OPERA-S1',
        processingLevel='CSLC',
        flightDirection = orbitPass,
        start=dateStart,
        end=dateEnd)

    ## Return results
    results = asf.search(**search_params)
    print(f"Length of Results: {len(results)}")

except TypeError:
    search_params = dict(
        intersectsWith= aoi.wkt,
        dataset='OPERA-S1',
        processingLevel='CSLC',
        flightDirection = orbitPass,
        start=dateStart,
        end=dateEnd)

    ## Return results
    results = asf.search(**search_params)
    print(f"Length of Results: {len(results)}")

In [ ]:
## Save the results in a geopandas dataframe
gf = gpd.GeoDataFrame.from_features(results.geojson(), crs='EPSG:4326')

## Filter data based on specified track number
gf = gf[gf.pathNumber==pathNumber]
# gf = gf[gf.pgeVersion=="2.1.1"] 
gf

In [ ]:
# Get only relevant metadata
cslc_df = gf[['operaBurstID', 'fileID', 'startTime', 'stopTime', 'url', 'geometry', 'pgeVersion']]
cslc_df['startTime'] = pd.to_datetime(cslc_df.startTime).dt.date
cslc_df['stopTime'] = pd.to_datetime(cslc_df.stopTime).dt.date

# Extract production time from fileID (2nd date token)
def _prod_time_from_fileid(file_id):
    # Example: OPERA_L2_CSLC-S1_..._20221122T161650Z_20240504T081640Z_...
    parts = str(file_id).split('_')
    return parts[5] if len(parts) > 5 else None

cslc_df['productionTime'] = pd.to_datetime(cslc_df['fileID'].apply(_prod_time_from_fileid), format='%Y%m%dT%H%M%SZ', errors='coerce')

# Keep newest duplicate by productionTime (fallback to pgeVersion, stopTime)
cslc_df = cslc_df.sort_values(by=['operaBurstID', 'startTime', 'productionTime', 'pgeVersion', 'stopTime'])
cslc_df = cslc_df.drop_duplicates(subset=['operaBurstID', 'startTime'], keep='last', ignore_index=True)

import re

def _subswath_from_fileid(file_id):
    # Example: ...-IW2_... -> IW2
    m = re.search(r"-IW[1-3]_", str(file_id))
    return m.group(0)[1:4] if m else None

cslc_df['burstSubswath'] = cslc_df['fileID'].apply(_subswath_from_fileid)

# Optional filtering by subswath or specific burst IDs
if BURST_SUBSWATH:
    if isinstance(BURST_SUBSWATH, (list, tuple, set)):
        subswaths = {str(s).upper() for s in BURST_SUBSWATH}
    else:
        subswaths = {str(BURST_SUBSWATH).upper()}
    cslc_df = cslc_df[cslc_df['burstSubswath'].str.upper().isin(subswaths)]

if BURST_ID:
    if isinstance(BURST_ID, (list, tuple, set)):
        burst_ids = {str(b) for b in BURST_ID}
    else:
        burst_ids = {str(BURST_ID)}
    cslc_df = cslc_df[cslc_df['operaBurstID'].isin(burst_ids)]
cslc_df


In [ ]:
import shapely.wkt as wkt
import geopandas as gpd

aoi_geom = wkt.loads(aoi)
aoi_gdf = gpd.GeoDataFrame(geometry=[aoi_geom], crs="EPSG:4326")

m = cslc_df[['operaBurstID', 'geometry']].explore(
    zoom=9,
    tiles="Esri.WorldImagery",
    style_kwds={"fill": True, "fillColor": "blue", "fillOpacity": 0.1, "weight": 2},
    name="CSLC footprints",
)

aoi_gdf.explore(
    m=m,
    color="red",
    style_kwds={"fill": True, "fillColor": "blue", "fillOpacity": 0.1, "weight": 2},
    name="AOI",
)

m


## 5. Download the CSLC-S1 locally

In [ ]:
## Download step skipped: CSLC subsets are streamed via opera-utils
print('Skipping full CSLC downloads; using opera-utils HTTP subsetting.')


In [ ]:
# Sort the CSLC-S1 by burstID and date
cslc_df = cslc_df.sort_values(by=["operaBurstID", "startTime"], ignore_index=True)
cslc_df


In [ ]:
# Enforce date range on dataframe (useful when re-running with narrower dates)
date_start_day = dateStart.date()
date_end_day = dateEnd.date()
cslc_df = cslc_df[(cslc_df['startTime'] >= date_start_day) & (cslc_df['startTime'] <= date_end_day)]
cslc_df = cslc_df.reset_index(drop=True)
cslc_df


## 6. Read each CSLC-S1 and stack them together


In [ ]:
cslc_stack = []; cslc_dates = []; bbox_stack = []; xcoor_stack = []; ycoor_stack = []

import os
import time
import numpy as np
import tempfile
import requests
import xarray as xr
import h5py
from tqdm.auto import tqdm
from pathlib import Path
from pyproj import Transformer
from shapely import wkt
from shapely.ops import transform as shp_transform
from opera_utils.credentials import get_earthdata_username_password
from opera_utils.disp._remote import open_file
from opera_utils.disp._utils import _get_netcdf_encoding

subset_dir = f"{savedir}/subset_cslc"
os.makedirs(subset_dir, exist_ok=True)

def _extract_subset(input_obj, outpath, rows, cols, chunks=(1,256,256)):
    X0, X1 = (cols.start, cols.stop) if cols is not None else (None, None)
    Y0, Y1 = (rows.start, rows.stop) if rows is not None else (None, None)
    ds = xr.open_dataset(input_obj, engine="h5netcdf", group="data")
    subset = ds.isel(y_coordinates=slice(Y0, Y1), x_coordinates=slice(X0, X1))
    subset.to_netcdf(
        outpath,
        engine="h5netcdf",
        group="data",
        encoding=_get_netcdf_encoding(subset, chunks=chunks),
    )
    for group in ("metadata", "identification"):
        with h5py.File(input_obj) as hf, h5py.File(outpath, "a") as dest_hf:
            hf.copy(group, dest_hf, name=group)
    with h5py.File(outpath, "a") as hf:
        ctype = h5py.h5t.py_create(np.complex64)
        ctype.commit(hf["/"].id, np.bytes_("complex64"))

def _subset_h5_to_disk(url, aoi_wkt, out_dir):
    outpath = Path(out_dir) / Path(url).name
    if outpath.exists():
        return outpath

    # determine row/col slices by reading coords
    with open_file(url) as in_f:
        ds = xr.open_dataset(in_f, engine="h5netcdf", group="data")
        xcoor = ds["x_coordinates"].values
        ycoor = ds["y_coordinates"].values
        epsg = int(ds["projection"].values)

    aoi_geom = wkt.loads(aoi_wkt)
    if epsg != 4326:
        transformer = Transformer.from_crs('EPSG:4326', f'EPSG:{epsg}', always_xy=True)
        aoi_geom = shp_transform(transformer.transform, aoi_geom)
    minx, miny, maxx, maxy = aoi_geom.bounds
    x_mask = (xcoor >= minx) & (xcoor <= maxx)
    y_mask = (ycoor >= miny) & (ycoor <= maxy)
    if not x_mask.any() or not y_mask.any():
        raise ValueError('AOI does not intersect this CSLC extent')
    ix = np.where(x_mask)[0]
    iy = np.where(y_mask)[0]
    rows = slice(iy.min(), iy.max()+1)
    cols = slice(ix.min(), ix.max()+1)

    if url.startswith('s3://'):
        with open_file(url) as in_f:
            _extract_subset(in_f, outpath, rows, cols)
    else:
        # HTTPS: download to temp then subset
        with tempfile.NamedTemporaryFile(suffix='.h5') as tf:
            if url.startswith('http'):
                session = requests.Session()
                username, password = get_earthdata_username_password()
                session.auth = (username, password)
                resp = session.get(url)
                resp.raise_for_status()
                tf.write(resp.content)
                tf.flush()
            _extract_subset(tf.name, outpath, rows, cols)
    return outpath

def _load_subset(file_id, url, start_date):
    outpath = _subset_h5_to_disk(url, aoi, subset_dir)
    # now read subset locally with h5py (fast)
    with h5py.File(outpath, 'r') as h5:
        cslc = h5['/data/VV'][:]
        xcoor = h5['/data/x_coordinates'][:]
        ycoor = h5['/data/y_coordinates'][:]
        dx = int(h5['/data/x_spacing'][()])
        dy = int(h5['/data/y_spacing'][()])
        epsg = int(h5['/data/projection'][()])
        sensing_start = h5['/metadata/processing_information/input_burst_metadata/sensing_start'][()].astype(str)
        sensing_stop = h5['/metadata/processing_information/input_burst_metadata/sensing_stop'][()].astype(str)
        dims = h5['/metadata/processing_information/input_burst_metadata/shape'][:]
        bounding_polygon = h5['/identification/bounding_polygon'][()].astype(str)
        orbit_direction = h5['/identification/orbit_pass_direction'][()].astype(str)
        center_lon, center_lat = h5['/metadata/processing_information/input_burst_metadata/center']
        wavelength = h5['/metadata/processing_information/input_burst_metadata/wavelength'][()].astype(str)
    subset_bbox = [float(xcoor.min()), float(xcoor.max()), float(ycoor.min()), float(ycoor.max())]
    return cslc, xcoor, ycoor, dx, dy, epsg, sensing_start, sensing_stop, dims, bounding_polygon, orbit_direction, center_lon, center_lat, wavelength, subset_bbox

# Subset with progress (parallel)
from concurrent.futures import ThreadPoolExecutor, as_completed

items = list(zip(cslc_df.fileID, cslc_df.url, cslc_df.startTime))
# Diagnostic: check pixel spacing before multilooking
with open_file(items[0][1]) as in_f:
    ds0 = xr.open_dataset(in_f, engine="h5netcdf", group="data")
    dx0 = float(ds0["x_spacing"].values)
    dy0 = float(ds0["y_spacing"].values)
print(f"Pixel spacing (dx, dy) = ({dx0}, {dy0})")

results = [None] * len(items)
_t0 = time.perf_counter()
with ThreadPoolExecutor(max_workers=DOWNLOAD_PROCESSES) as ex:
    futures = {ex.submit(_load_subset, fileID, url, start_date): i for i, (fileID, url, start_date) in enumerate(items)}
    for fut in tqdm(as_completed(futures), total=len(futures), desc='Subsetting CSLC'):
        i = futures[fut]
        results[i] = fut.result()
_t1 = time.perf_counter()
print(f"Subset/download time: {_t1 - _t0:.1f} s")

for (fileID, start_date), res in zip(zip(cslc_df.fileID, cslc_df.startTime), results):
    cslc, xcoor, ycoor, dx, dy, epsg, sensing_start, sensing_stop, dims, bounding_polygon, orbit_direction, center_lon, center_lat, wavelength, subset_bbox = res
    cslc_stack.append(cslc)
    cslc_dates.append(pd.to_datetime(sensing_start).date())
    if subset_bbox is not None:
        bbox = subset_bbox
    else:
        cslc_poly = wkt.loads(bounding_polygon)
        bbox = [cslc_poly.bounds[0], cslc_poly.bounds[2], cslc_poly.bounds[1], cslc_poly.bounds[3]]
    bbox_stack.append(bbox)
    xcoor_stack.append(xcoor)
    ycoor_stack.append(ycoor)


## 7. Generate the interferograms, compute for the coherence, save the files as GeoTiffs

In [ ]:
def colorize(array=[], cmap='RdBu', cmin=[], cmax=[]):
    normed_data = (array - cmin) / (cmax - cmin)    
    cm = plt.cm.get_cmap(cmap)
    return cm(normed_data) 

In [ ]:
def goldstein_filter(ifg_cpx, alpha=0.5, pad=32, edge_trim=16):
    # Goldstein filter with padding + taper + mask to reduce edge effects
    mask = np.isfinite(ifg_cpx)
    data = np.nan_to_num(ifg_cpx, nan=0.0)
    if pad and pad > 0:
        data = np.pad(data, ((pad, pad), (pad, pad)), mode="reflect")
        mask = np.pad(mask, ((pad, pad), (pad, pad)), mode="constant", constant_values=False)
    # Apply 2D Hann window (taper)
    wy = np.hanning(data.shape[0])
    wx = np.hanning(data.shape[1])
    window = wy[:, None] * wx[None, :]
    f = np.fft.fft2(data * window)
    s = np.abs(f)
    s = s / (s.max() + 1e-8)
    f_filt = f * (s ** alpha)
    out = np.fft.ifft2(f_filt)
    if pad and pad > 0:
        out = out[pad:-pad, pad:-pad]
        mask = mask[pad:-pad, pad:-pad]
    # restore NaNs outside valid mask
    out[~mask] = np.nan
    if edge_trim and edge_trim > 0:
        out[:edge_trim, :] = np.nan
        out[-edge_trim:, :] = np.nan
        out[:, :edge_trim] = np.nan
        out[:, -edge_trim:] = np.nan
    return out


In [ ]:
def rasterWrite(outtif,arr,transform,epsg,dtype='float32'):
    #writing geotiff using rasterio
    
    new_dataset = rasterio.open(outtif, 'w', driver='GTiff',
                            height = arr.shape[0], width = arr.shape[1],
                            count=1, dtype=dtype,
                            crs=CRS.from_epsg(epsg),
                            transform=transform,nodata=np.nan)
    new_dataset.write(arr, 1)
    new_dataset.close() 

In [ ]:
## Build date pairs per burstID
cslc_dates = cslc_df[["startTime"]]
burstID = cslc_df.operaBurstID.drop_duplicates(ignore_index=True)
n_unique_burstID = len(burstID)

def _lag_list(lag):
    if lag is None:
        return []
    if isinstance(lag, (list, tuple, set)):
        return sorted({int(x) for x in lag})
    return [int(lag)]

pair_lags = _lag_list(pair_t_span_days)
pair_indices = []  # list of (ref_idx, sec_idx) in cslc_df order

for bid, group in cslc_df.groupby('operaBurstID'):
    group = group.sort_values('startTime')
    idx = group.index.to_list()
    dates = group['startTime'].to_list()

    burst_pairs = []
    if pair_mode == 'all':
        for i in range(len(idx)):
            for j in range(i+1, len(idx)):
                delta = (dates[j] - dates[i]).days
                if delta < min_t_span_days:
                    continue
                if max_t_span_days is not None and delta > max_t_span_days:
                    continue
                burst_pairs.append((idx[i], idx[j]))
    elif pair_mode == 't_span':
        for i in range(len(idx)):
            for j in range(i+1, len(idx)):
                delta = (dates[j] - dates[i]).days
                if delta in pair_lags and delta >= min_t_span_days and (max_t_span_days is None or delta <= max_t_span_days):
                    burst_pairs.append((idx[i], idx[j]))
    else:
        raise ValueError("pair_mode must be 'all' or 't_span'")

    if max_pairs_per_burst is not None:
        burst_pairs = burst_pairs[:int(max_pairs_per_burst)]

    pair_indices.extend(burst_pairs)

if max_pairs_total is not None:
    pair_indices = pair_indices[:int(max_pairs_total)]

# Sort pairs by date, then burstID (so same dates group together)
def _pair_sort_key(pair):
    ref_idx, sec_idx = pair
    ref_date = cslc_dates.iloc[ref_idx].values[0]
    sec_date = cslc_dates.iloc[sec_idx].values[0]
    burst = cslc_df.operaBurstID.iloc[ref_idx]
    return (ref_date, sec_date, burst)
pair_indices = sorted(pair_indices, key=_pair_sort_key)
print(f'Pair count: {len(pair_indices)}')


In [ ]:
import numpy as np
import warnings
from numpy.lib.stride_tricks import sliding_window_view




def take_looks(arr, row_looks, col_looks, func_type="nanmean", edge_strategy="cutoff"):
    if row_looks == 1 and col_looks == 1:
        return arr
    if arr.ndim != 2:
        raise ValueError("take_looks expects 2D array")
    rows, cols = arr.shape
    if edge_strategy == "cutoff":
        rows = (rows // row_looks) * row_looks
        cols = (cols // col_looks) * col_looks
        arr = arr[:rows, :cols]
    elif edge_strategy == "pad":
        pad_r = (-rows) % row_looks
        pad_c = (-cols) % col_looks
        if pad_r or pad_c:
            arr = np.pad(arr, ((0, pad_r), (0, pad_c)), mode="constant", constant_values=np.nan)
        rows, cols = arr.shape
    else:
        raise ValueError("edge_strategy must be 'cutoff' or 'pad'")

    new_rows = rows // row_looks
    new_cols = cols // col_looks
    func = getattr(np, func_type)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        return func(arr.reshape(new_rows, row_looks, new_cols, col_looks), axis=(1, 3))


def _multilook(arr, looks_y=1, looks_x=1):
    return take_looks(arr, looks_y, looks_x, func_type="nanmean", edge_strategy="cutoff")

def _box_mean(arr, win):
    pad = win // 2
    arr_p = np.pad(arr, pad_width=pad, mode='reflect')
    windows = sliding_window_view(arr_p, (win, win))
    return windows.mean(axis=(-2, -1))

    return _box_mean(arr, win)

def lee_filter(img, win=5):
    mean = _box_mean(img, win)
    mean_sq = _box_mean(img**2, win)
    var = mean_sq - mean**2
    noise_var = np.nanmedian(var)
    w = var / (var + noise_var + 1e-8)
    return mean + w * (img - mean)

def goldstein(phase, alpha, psize=32):
    """Apply the Goldstein adaptive filter to the given data."""
    def apply_pspec(data):
        if alpha < 0:
            raise ValueError(f"alpha must be >= 0, got {alpha = }")
        weight = np.power(np.abs(data) ** 2, alpha / 2)
        data = weight * data
        return data

    def make_weight(nxp, nyp):
        wx = 1.0 - np.abs(np.arange(nxp // 2) - (nxp / 2.0 - 1.0)) / (nxp / 2.0 - 1.0)
        wy = 1.0 - np.abs(np.arange(nyp // 2) - (nyp / 2.0 - 1.0)) / (nyp / 2.0 - 1.0)
        quadrant = np.outer(wy, wx)
        weight = np.block(
            [
                [quadrant, np.flip(quadrant, axis=1)],
                [np.flip(quadrant, axis=0), np.flip(np.flip(quadrant, axis=0), axis=1)],
            ]
        )
        return weight

    def patch_goldstein_filter(data, weight, psize):
        data = np.fft.fft2(data, s=(psize, psize))
        data = apply_pspec(data)
        data = np.fft.ifft2(data, s=(psize, psize))
        return weight * data

    def apply_goldstein_filter(data):
        out = np.zeros(data.shape, dtype=np.complex64)
        empty_mask = np.isnan(data) | (np.angle(data) == 0)
        if np.all(empty_mask):
            return data
        weight_matrix = make_weight(psize, psize)
        for i in range(0, data.shape[0] - psize, psize // 2):
            for j in range(0, data.shape[1] - psize, psize // 2):
                data_window = data[i : i + psize, j : j + psize]
                weight_window = weight_matrix[: data_window.shape[0], : data_window.shape[1]]
                filtered_window = patch_goldstein_filter(data_window, weight_window, psize)
                slice_i = slice(i, min(i + psize, out.shape[0]))
                slice_j = slice(j, min(j + psize, out.shape[1]))
                out[slice_i, slice_j] += filtered_window[: slice_i.stop - slice_i.start, : slice_j.stop - slice_j.start]
        out[empty_mask] = 0
        return out

    if np.iscomplexobj(phase):
        return apply_goldstein_filter(phase)
    else:
        return apply_goldstein_filter(np.exp(1j * phase))

    phase = reference * np.conjugate(secondary)
    amp = np.sqrt((reference * np.conjugate(reference)) * (secondary * np.conjugate(secondary)))
    nan_mask = np.isnan(phase)
    ifg[nan_mask] = np.nan
    ifg_cpx = np.exp(1j * np.nan_to_num(np.angle(phase/amp)))
    zero_mask = phase == 0
    coh[nan_mask] = np.nan
    coh[zero_mask] = 0
    return ifg, coh, amp

def calc_ifg_coh_filtered(reference, secondary, goldstein_alpha=0.5, coh_win=5, looks_y=1, looks_x=1):
    reference = _multilook(reference, looks_y, looks_x)
    secondary = _multilook(secondary, looks_y, looks_x)
    phase = reference * np.conjugate(secondary)
    amp = np.sqrt((reference * np.conjugate(reference)) * (secondary * np.conjugate(secondary)))
    nan_mask = np.isnan(phase)
    ifg_cpx = np.exp(1j * np.nan_to_num(np.angle(phase/amp)))
    ifg_cpx_f = goldstein(ifg_cpx, alpha=goldstein_alpha, psize=32)
    ifg = np.angle(ifg_cpx_f)
    ifg[nan_mask] = np.nan
    coh = np.abs(_box_mean(ifg_cpx, coh_win))
    coh = np.clip(coh, 0, 1)
    coh = lee_filter(coh, win=coh_win)
    coh = np.clip(coh, 0, 1)
    zero_mask = phase == 0
    coh[nan_mask] = np.nan
    coh[zero_mask] = 0
    return ifg, coh, amp

def calc_ifg_coh(reference, secondary, looks_y=1, looks_x=1):
    return calc_ifg_coh_filtered(reference, secondary, looks_y=looks_y, looks_x=looks_x)


In [ ]:
## For each date-pair, calculate the ifg, coh. Save the results as GeoTiffs.
for ref_idx, sec_idx in pair_indices:
    ref_date = cslc_dates.iloc[ref_idx].values[0]
    sec_date = cslc_dates.iloc[sec_idx].values[0]
    print(f"Reference: {ref_date}  Secondary: {sec_date}")

    # Calculate ifg, coh, amp
    if "calc_ifg_coh_filtered" not in globals():
        raise RuntimeError("calc_ifg_coh_filtered is not defined. Run the filter definition cell first.")
    looks_y, looks_x = MULTILOOK
    ifg, coh, amp = calc_ifg_coh_filtered(cslc_stack[ref_idx], cslc_stack[sec_idx], looks_y=looks_y, looks_x=looks_x)

    # Save each interferogram as GeoTiff (no per-burst plotting)
    transform = from_origin(xcoor_stack[ref_idx][0], ycoor_stack[ref_idx][0], dx, np.abs(dy))


## 8. Merge the burst-wise interferograms and coherence and save as GeoTiff.

In [ ]:
def custom_merge(old_data, new_data, old_nodata, new_nodata, **kwargs):    
    mask = np.logical_and(~old_nodata, ~new_nodata)
    old_data[mask] = new_data[mask]
    mask = np.logical_and(old_nodata, ~new_nodata)
    old_data[mask] = new_data[mask]


In [ ]:
os.makedirs(f"{savedir}/tifs", exist_ok=True)
# Merge burst-wise interferograms per date-pair (in-memory)
from rasterio.io import MemoryFile

from rasterio.warp import calculate_default_transform, reproject, Resampling


from rasterio.warp import reproject, Resampling
from pathlib import Path

def _load_water_mask_match(mask_path, shape, transform, crs):
    if mask_path is None or not Path(mask_path).exists():
        return None
    with rasterio.open(mask_path) as src:
        src_mask = src.read(1)
        if src.crs == crs and src.transform == transform and src_mask.shape == shape:
            return src_mask
        dst = np.zeros(shape, dtype=src_mask.dtype)
        reproject(
            source=src_mask,
            destination=dst,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=transform,
            dst_crs=crs,
            resampling=Resampling.nearest,
        )
        return dst

def _apply_water_mask(arr, mask):
    # mask: 1 = keep land, 0 = water
    return np.where(mask == 0, np.nan, arr)


from affine import Affine

def _trim_nan_border(arr, transform):
    data = arr[0] if arr.ndim == 3 else arr
    mask = np.isfinite(data) & (data != 0)
    if not mask.any():
        return arr, transform
    rows = np.where(mask.any(axis=1))[0]
    cols = np.where(mask.any(axis=0))[0]
    r0, r1 = rows[0], rows[-1] + 1
    c0, c1 = cols[0], cols[-1] + 1
    data = data[r0:r1, c0:c1]
    if arr.ndim == 3:
        arr = data[None, ...]
    else:
        arr = data
    new_transform = transform * Affine.translation(c0, r0)
    return arr, new_transform

def _save_mosaic_utm_to_wgs84(out_path, mosaic, transform, epsg):
    import os
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    dst_crs = 'EPSG:4326'
    dst_transform, width, height = calculate_default_transform(
        f'EPSG:{epsg}', dst_crs, mosaic.shape[2], mosaic.shape[1], *rasterio.transform.array_bounds(mosaic.shape[1], mosaic.shape[2], transform)
    )
    dest = np.zeros((1, height, width), dtype=mosaic.dtype)
    reproject(
        source=mosaic,
        destination=dest,
        src_transform=transform,
        src_crs=f'EPSG:{epsg}',
        dst_transform=dst_transform,
        dst_crs=dst_crs,
        resampling=Resampling.nearest,
    )
    out_meta = {
        'driver': 'GTiff',
        'height': height,
        'width': width,
        'count': 1,
        'dtype': mosaic.dtype,
        'crs': dst_crs,
        'transform': dst_transform,
    }
    with rasterio.open(out_path, 'w', **out_meta) as dst:
        dst.write(dest)

# Group pair indices by date tag
pairs_by_tag = {}
for r, s in pair_indices:
    ref_date = cslc_dates.iloc[r].values[0]
    sec_date = cslc_dates.iloc[s].values[0]
    tag = f"{ref_date.strftime('%Y%m%d')}-{sec_date.strftime('%Y%m%d')}"
    pairs_by_tag.setdefault(tag, []).append((r, s))

for tag, pairs in pairs_by_tag.items():
    srcs = []
    for r, s in pairs:
        looks_y, looks_x = MULTILOOK
        ifg, coh, amp = calc_ifg_coh(cslc_stack[r], cslc_stack[s], looks_y=looks_y, looks_x=looks_x)
        dy_signed = (ycoor_stack[r][1] - ycoor_stack[r][0]) if len(ycoor_stack[r]) > 1 else -dy
        x0 = xcoor_stack[r][0] + (looks_x - 1) * dx / 2
        y0 = ycoor_stack[r][0] + (looks_y - 1) * dy_signed / 2
        transform = from_origin(x0, y0, dx*looks_x, np.abs(dy_signed)*looks_y)
        mem = MemoryFile()
        ds = mem.open(
            driver='GTiff', height=ifg.shape[0], width=ifg.shape[1], count=1, dtype=ifg.dtype,
            crs=CRS.from_epsg(epsg), transform=transform, nodata=np.nan
        )
        ds.write(ifg, 1)
        srcs.append(ds)
    dest, output_transform = merge.merge(srcs, method=custom_merge)
    dest, output_transform = _trim_nan_border(dest, output_transform)
    if APPLY_WATER_MASK and WATER_MASK_PATH:
        mask = _load_water_mask_match(WATER_MASK_PATH, dest.shape[1:], output_transform, CRS.from_epsg(epsg))
        if mask is not None:
            dest[0] = _apply_water_mask(dest[0], mask)
    out_meta = srcs[0].meta.copy()
    out_meta.update({"driver": "GTiff", "height": dest.shape[1], "width": dest.shape[2], "transform": output_transform})
    out_path = f"{savedir}/tifs/merged_ifg_{tag}.tif"
    with rasterio.open(out_path, "w", **out_meta) as dest1:
        dest1.write(dest)
    if SAVE_WGS84:
        out_path_wgs84 = f"{savedir}/tifs/WGS84/merged_ifg_WGS84_{tag}.tif"
        _save_mosaic_utm_to_wgs84(out_path_wgs84, dest, output_transform, epsg)
    for ds in srcs:
        ds.close()


In [ ]:
os.makedirs(f"{savedir}/tifs", exist_ok=True)
# Merge burst-wise coherence per date-pair (in-memory)
from rasterio.io import MemoryFile

from rasterio.warp import calculate_default_transform, reproject, Resampling


from rasterio.warp import reproject, Resampling

def _load_water_mask_match(mask_path, shape, transform, crs):
    with rasterio.open(mask_path) as src:
        src_mask = src.read(1)
        if src.crs == crs and src.transform == transform and src_mask.shape == shape:
            return src_mask
        dst = np.zeros(shape, dtype=src_mask.dtype)
        reproject(
            source=src_mask,
            destination=dst,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=transform,
            dst_crs=crs,
            resampling=Resampling.nearest,
        )
        return dst

def _apply_water_mask(arr, mask):
    # mask: 1 = keep land, 0 = water
    return np.where(mask == 0, np.nan, arr)


from affine import Affine

def _trim_nan_border(arr, transform):
    data = arr[0] if arr.ndim == 3 else arr
    mask = np.isfinite(data) & (data != 0)
    if not mask.any():
        return arr, transform
    rows = np.where(mask.any(axis=1))[0]
    cols = np.where(mask.any(axis=0))[0]
    r0, r1 = rows[0], rows[-1] + 1
    c0, c1 = cols[0], cols[-1] + 1
    data = data[r0:r1, c0:c1]
    if arr.ndim == 3:
        arr = data[None, ...]
    else:
        arr = data
    new_transform = transform * Affine.translation(c0, r0)
    return arr, new_transform

def _save_mosaic_utm_to_wgs84(out_path, mosaic, transform, epsg):
    import os
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    dst_crs = 'EPSG:4326'
    dst_transform, width, height = calculate_default_transform(
        f'EPSG:{epsg}', dst_crs, mosaic.shape[2], mosaic.shape[1], *rasterio.transform.array_bounds(mosaic.shape[1], mosaic.shape[2], transform)
    )
    dest = np.zeros((1, height, width), dtype=mosaic.dtype)
    reproject(
        source=mosaic,
        destination=dest,
        src_transform=transform,
        src_crs=f'EPSG:{epsg}',
        dst_transform=dst_transform,
        dst_crs=dst_crs,
        resampling=Resampling.nearest,
    )
    out_meta = {
        'driver': 'GTiff',
        'height': height,
        'width': width,
        'count': 1,
        'dtype': mosaic.dtype,
        'crs': dst_crs,
        'transform': dst_transform,
    }
    with rasterio.open(out_path, 'w', **out_meta) as dst:
        dst.write(dest)

# Group pair indices by date tag
pairs_by_tag = {}
for r, s in pair_indices:
    ref_date = cslc_dates.iloc[r].values[0]
    sec_date = cslc_dates.iloc[s].values[0]
    tag = f"{ref_date.strftime('%Y%m%d')}-{sec_date.strftime('%Y%m%d')}"
    pairs_by_tag.setdefault(tag, []).append((r, s))

for tag, pairs in pairs_by_tag.items():
    srcs = []
    for r, s in pairs:
        looks_y, looks_x = MULTILOOK
        ifg, coh, amp = calc_ifg_coh(cslc_stack[r], cslc_stack[s], looks_y=looks_y, looks_x=looks_x)
        dy_signed = (ycoor_stack[r][1] - ycoor_stack[r][0]) if len(ycoor_stack[r]) > 1 else -dy
        x0 = xcoor_stack[r][0] + (looks_x - 1) * dx / 2
        y0 = ycoor_stack[r][0] + (looks_y - 1) * dy_signed / 2
        transform = from_origin(x0, y0, dx*looks_x, np.abs(dy_signed)*looks_y)
        mem = MemoryFile()
        ds = mem.open(
            driver='GTiff', height=coh.shape[0], width=coh.shape[1], count=1, dtype=coh.dtype,
            crs=CRS.from_epsg(epsg), transform=transform, nodata=np.nan
        )
        ds.write(coh, 1)
        srcs.append(ds)
    dest, output_transform = merge.merge(srcs, method=custom_merge)
    dest, output_transform = _trim_nan_border(dest, output_transform)
    if APPLY_WATER_MASK and WATER_MASK_PATH:
        mask = _load_water_mask_match(WATER_MASK_PATH, dest.shape[1:], output_transform, CRS.from_epsg(epsg))
        dest[0] = _apply_water_mask(dest[0], mask)
    out_meta = srcs[0].meta.copy()
    out_meta.update({"driver": "GTiff", "height": dest.shape[1], "width": dest.shape[2], "transform": output_transform})
    out_path = f"{savedir}/tifs/merged_coh_{tag}.tif"
    with rasterio.open(out_path, "w", **out_meta) as dest1:
        dest1.write(dest)
    if SAVE_WGS84:
        out_path_wgs84 = f"{savedir}/tifs/WGS84/merged_coh_WGS84_{tag}.tif"
        _save_mosaic_utm_to_wgs84(out_path_wgs84, dest, output_transform, epsg)
    for ds in srcs:
        ds.close()


## 9. Read the merged GeoTiff and Visualize using `matplotlib`

In [ ]:
# Read merged IFG/COH files and plot paired grids
import glob
import math
import pandas as pd
import os


# Output dir for per-pair PNGs
pair_png_dir = f"{savedir}/pairs_png"
os.makedirs(pair_png_dir, exist_ok=True)

ifg_paths = sorted(glob.glob(f"{savedir}/tifs/merged_ifg_*.tif"))
coh_paths = sorted(glob.glob(f"{savedir}/tifs/merged_coh_*.tif"))

ifg_map = {p.split('merged_ifg_')[-1].replace('.tif',''): p for p in ifg_paths}
coh_map = {p.split('merged_coh_')[-1].replace('.tif',''): p for p in coh_paths}



def _prep_da(path):
    da = rioxarray.open_rasterio(path)[0]
    if REPROJECT_FOR_DISPLAY:
        da = da.rio.reproject("EPSG:4326")
    data = da.values
    mask = np.isfinite(data) & (data != 0)
    if mask.any():
        rows = np.where(mask.any(axis=1))[0]
        cols = np.where(mask.any(axis=0))[0]
        r0, r1 = rows[0], rows[-1] + 1
        c0, c1 = cols[0], cols[-1] + 1
        # Trim NaN borders so edges don't show padding
        da = da.isel(y=slice(r0, r1), x=slice(c0, c1))
    return da

pair_tags = sorted(set(ifg_map).intersection(coh_map))
# Filter pairs by current date range
date_start_day = dateStart.date()
date_end_day = dateEnd.date()
pair_tags = [t for t in pair_tags if (date_start_day <= pd.to_datetime(t.split('-')[0], format='%Y%m%d').date() <= date_end_day and date_start_day <= pd.to_datetime(t.split('-')[1], format='%Y%m%d').date() <= date_end_day)]

if not pair_tags:
    print('No matching IFG/COH pairs found')
else:
    # Save ALL pairs as PNGs
    for tag in pair_tags:
        fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
        ax_ifg, ax_coh = axes

        # IFG
        merged_ifg = _prep_da(ifg_map[tag])
        minlon, minlat, maxlon, maxlat = merged_ifg.rio.bounds()
        bbox = [minlon, maxlon, minlat, maxlat]
        colored_ifg = colorize(merged_ifg, 'twilight_shifted', -np.pi, np.pi)
        colored_ifg = np.ma.masked_invalid(colored_ifg)
        im_ifg = ax_ifg.imshow(colored_ifg, cmap='twilight_shifted', interpolation='none', origin='upper', extent=bbox, vmin=-np.pi, vmax=np.pi)
        ax_ifg.set_title(f"IFG_{tag}", fontsize=10)
        ax_ifg.set_xticks([])
        ax_ifg.set_yticks([])
        fig.colorbar(im_ifg, ax=ax_ifg, orientation='vertical', fraction=0.046, pad=0.02, label='Wrapped phase (rad)')

        # COH
        merged_coh = _prep_da(coh_map[tag])
        minlon, minlat, maxlon, maxlat = merged_coh.rio.bounds()
        bbox = [minlon, maxlon, minlat, maxlat]
        coh_vals = np.ma.masked_invalid(merged_coh.values)
        im_coh = ax_coh.imshow(coh_vals, cmap='gray', interpolation='none', origin='upper', extent=bbox, vmin=0, vmax=1.0)
        ax_coh.set_title(f"COH_{tag}", fontsize=10)
        ax_coh.set_xticks([])
        ax_coh.set_yticks([])
        fig.colorbar(im_coh, ax=ax_coh, orientation='vertical', fraction=0.046, pad=0.02, label='Coherence')

        out_png = os.path.join(pair_png_dir, f"pair_{tag}.png")
        fig.savefig(out_png, dpi=150)
        plt.close(fig)

    # Display only last 5 pairs in notebook
    display_tags = pair_tags[-5:]
    n = len(display_tags)
    ncols = 2
    nrows = math.ceil(n / 1)  # one pair per row
    fig, axes = plt.subplots(nrows, ncols, figsize=(6*ncols, 3*nrows), constrained_layout=True)
    if nrows == 1:
        axes = [axes]

    for i, tag in enumerate(display_tags):
        ax_ifg, ax_coh = axes[i]

        # IFG
        merged_ifg = _prep_da(ifg_map[tag])
        minlon, minlat, maxlon, maxlat = merged_ifg.rio.bounds()
        bbox = [minlon, maxlon, minlat, maxlat]
        colored_ifg = colorize(merged_ifg, 'twilight_shifted', -np.pi, np.pi)
        colored_ifg = np.ma.masked_invalid(colored_ifg)
        im_ifg = ax_ifg.imshow(colored_ifg, cmap='twilight_shifted', interpolation='none', origin='upper', extent=bbox, vmin=-np.pi, vmax=np.pi)
        ax_ifg.set_title(f"IFG_{tag}", fontsize=10)
        ax_ifg.set_xticks([])
        ax_ifg.set_yticks([])
        fig.colorbar(im_ifg, ax=ax_ifg, orientation='vertical', fraction=0.046, pad=0.02, label='Wrapped phase (rad)')

        # COH
        merged_coh = _prep_da(coh_map[tag])
        minlon, minlat, maxlon, maxlat = merged_coh.rio.bounds()
        bbox = [minlon, maxlon, minlat, maxlat]
        coh_vals = np.ma.masked_invalid(merged_coh.values)
        im_coh = ax_coh.imshow(coh_vals, cmap='gray', interpolation='none', origin='upper', extent=bbox, vmin=0, vmax=1.0)
        ax_coh.set_title(f"COH_{tag}", fontsize=10)
        ax_coh.set_xticks([])
        ax_coh.set_yticks([])
        fig.colorbar(im_coh, ax=ax_coh, orientation='vertical', fraction=0.046, pad=0.02, label='Coherence')


## 9.5 Merge and plot amplitude mosaics (per date)


In [ ]:
import h5py
import numpy as np
import glob
import math
import rasterio
from rasterio.transform import from_origin
from rasterio.crs import CRS
from rasterio.warp import calculate_default_transform, reproject, Resampling


from rasterio.warp import reproject, Resampling

def _load_water_mask_match(mask_path, shape, transform, crs):
    with rasterio.open(mask_path) as src:
        src_mask = src.read(1)
        if src.crs == crs and src.transform == transform and src_mask.shape == shape:
            return src_mask
        dst = np.zeros(shape, dtype=src_mask.dtype)
        reproject(
            source=src_mask,
            destination=dst,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=transform,
            dst_crs=crs,
            resampling=Resampling.nearest,
        )
        return dst

def _apply_water_mask(arr, mask):
    # mask: 1 = keep land, 0 = water
    return np.where(mask == 0, np.nan, arr)


from affine import Affine

def _trim_nan_border(arr, transform):
    data = arr[0] if arr.ndim == 3 else arr
    mask = np.isfinite(data) & (data != 0)
    if not mask.any():
        return arr, transform
    rows = np.where(mask.any(axis=1))[0]
    cols = np.where(mask.any(axis=0))[0]
    r0, r1 = rows[0], rows[-1] + 1
    c0, c1 = cols[0], cols[-1] + 1
    data = data[r0:r1, c0:c1]
    if arr.ndim == 3:
        arr = data[None, ...]
    else:
        arr = data
    new_transform = transform * Affine.translation(c0, r0)
    return arr, new_transform

# Build per-date amplitude mosaics directly from subset H5 (no per-burst GeoTIFFs)
os.makedirs(f"{savedir}/tifs", exist_ok=True)

date_tags = sorted(cslc_df.startTime.astype(str).str.replace('-', '').unique())

def _save_mosaic_utm(out_path, mosaic, transform, epsg):
    with rasterio.open(
        out_path, "w", driver="GTiff", height=mosaic.shape[0], width=mosaic.shape[1],
        count=1, dtype=mosaic.dtype, crs=CRS.from_epsg(epsg), transform=transform, nodata=np.nan
    ) as dst_ds:
        dst_ds.write(mosaic, 1)

def _save_mosaic_utm_to_wgs84(out_path, mosaic, transform, epsg):
    dst_crs = "EPSG:4326"
    src_crs = CRS.from_epsg(epsg)
    height, width = mosaic.shape
    dst_transform, dst_width, dst_height = calculate_default_transform(
        src_crs, dst_crs, width, height, *rasterio.transform.array_bounds(height, width, transform)
    )
    dst = np.empty((dst_height, dst_width), dtype=mosaic.dtype)
    reproject(
        source=mosaic,
        destination=dst,
        src_transform=transform,
        src_crs=src_crs,
        dst_transform=dst_transform,
        dst_crs=dst_crs,
        resampling=Resampling.bilinear,
    )
    with rasterio.open(
        out_path, "w", driver="GTiff", height=dst_height, width=dst_width, count=1,
        dtype=dst.dtype, crs=dst_crs, transform=dst_transform, nodata=np.nan
    ) as dst_ds:
        dst_ds.write(dst, 1)

# Mosaicking helper (in memory)
def _mosaic_arrays(arrays, transforms, epsg):
    # Convert arrays to in-memory rasterio datasets via MemoryFile
    from rasterio.io import MemoryFile
    srcs = []
    for arr, transform in zip(arrays, transforms):
        mem = MemoryFile()
        ds = mem.open(
            driver='GTiff', height=arr.shape[0], width=arr.shape[1], count=1, dtype=arr.dtype,
            crs=CRS.from_epsg(epsg), transform=transform, nodata=np.nan
        )
        ds.write(arr, 1)
        srcs.append(ds)
    dest, out_transform = merge.merge(srcs, method=custom_merge)
    for ds in srcs:
        ds.close()
    return dest[0], out_transform

looks_y, looks_x = MULTILOOK
for date_tag in date_tags:
    # collect subset H5 files for this date
    rows = cslc_df[cslc_df.startTime.astype(str).str.replace('-', '') == date_tag]
    arrays = []
    transforms = []
    epsg = None
    for fileID in rows.fileID:
        subset_path = f"{savedir}/subset_cslc/{fileID}.h5"
        with h5py.File(subset_path, 'r') as h5:
            cslc = h5['/data/VV'][:]
            xcoor = h5['/data/x_coordinates'][:]
            ycoor = h5['/data/y_coordinates'][:]
            dx = int(h5['/data/x_spacing'][()])
            dy = int(h5['/data/y_spacing'][()])
            epsg = int(h5['/data/projection'][()])
        power_ml = _multilook(np.abs(cslc)**2, looks_y, looks_x)
        amp = 10*np.log10(power_ml)
        dy_signed = (ycoor[1] - ycoor[0]) if len(ycoor) > 1 else -dy
        x0 = xcoor[0] + (looks_x - 1) * dx / 2
        y0 = ycoor[0] + (looks_y - 1) * dy_signed / 2
        transform = from_origin(x0, y0, dx*looks_x, np.abs(dy_signed)*looks_y)
        arrays.append(amp)
        transforms.append(transform)

    if not arrays:
        continue
    mosaic, out_transform = _mosaic_arrays(arrays, transforms, epsg)
    out_path_utm = f"{savedir}/tifs/merged_amp_{date_tag}.tif"
    mosaic, out_transform = _trim_nan_border(mosaic, out_transform)
    if APPLY_WATER_MASK and WATER_MASK_PATH:
        mask = _load_water_mask_match(WATER_MASK_PATH, mosaic.shape, out_transform, CRS.from_epsg(epsg))
        mosaic = _apply_water_mask(mosaic, mask)
    _save_mosaic_utm(out_path_utm, mosaic, out_transform, epsg)
    if SAVE_WGS84:
        out_path_wgs84 = f"{savedir}/tifs/WGS84/merged_amp_WGS84_{date_tag}.tif"
        _save_mosaic_utm_to_wgs84(out_path_wgs84, mosaic, out_transform, epsg)

# Plot merged amplitude mosaics in a grid (native CRS from saved GeoTIFFs)

# Output dir for amplitude PNGs
amp_png_dir = f"{savedir}/amp_png"
os.makedirs(amp_png_dir, exist_ok=True)

paths = sorted(glob.glob(f"{savedir}/tifs/merged_amp_*.tif"))
paths = [p for p in paths if 'WGS84' not in p]
all_vals = []
for p in paths:
    da = rioxarray.open_rasterio(p)[0]
    all_vals.append(da.values.ravel())
if all_vals:
    all_vals = np.concatenate(all_vals)
    gmin = np.nanpercentile(all_vals, 2)
    gmax = np.nanpercentile(all_vals, 90)
else:
    gmin, gmax = None, None
n = len(paths)
if n == 0:
    print('No merged amplitude files found')
else:
    # Save ALL amplitude PNGs
    for path in paths:
        src = rioxarray.open_rasterio(path)
        amp = src[0]
        minlon, minlat, maxlon, maxlat = amp.rio.bounds()
        bbox = [minlon, maxlon, minlat, maxlat]
        fig, ax = plt.subplots(figsize=(5,4))
        im = ax.imshow(amp.values, cmap='gray', interpolation='none', origin='upper', extent=bbox, vmin=gmin, vmax=gmax)
        tag = path.split('merged_amp_')[-1].replace('.tif','')
        ax.set_title(f"AMP_{tag}", fontsize=10)
        ax.set_xticks([])
        ax.set_yticks([])
        fig.colorbar(im, ax=ax, orientation='vertical', fraction=0.046, pad=0.02)
        out_png = os.path.join(amp_png_dir, f"amp_{tag}.png")
        fig.savefig(out_png, dpi=150)
        plt.close(fig)

    # Show only last 5 in notebook
    display_paths = paths[-5:]
    n = len(display_paths)
    ncols = 3
    nrows = math.ceil(n / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(4*ncols, 3*nrows), constrained_layout=True)
    axes = axes.ravel()
    for ax, path in zip(axes, display_paths):
        src = rioxarray.open_rasterio(path)
        amp = src[0]
        minlon, minlat, maxlon, maxlat = amp.rio.bounds()
        bbox = [minlon, maxlon, minlat, maxlat]
        im = ax.imshow(amp.values, cmap='gray', interpolation='none', origin='upper', extent=bbox, vmin=gmin, vmax=gmax)
        tag = path.split('merged_amp_')[-1].replace('.tif','')
        ax.set_title(f"AMP_{tag}", fontsize=10)
        ax.set_xticks([])
        ax.set_yticks([])
    for ax in axes[n:]:
        ax.axis('off')
    fig.colorbar(im, ax=axes.tolist(), orientation='vertical', fraction=0.02, pad=0.02)


## 10. Monthly mean coherence calendar (per year)


In [ ]:
import glob
import pandas as pd
import numpy as np
import rioxarray
import xarray as xr
import matplotlib.pyplot as plt



# Build an index of merged coherence files by midpoint year-month
records = []
for path in sorted(glob.glob(f"{savedir}/tifs/merged_coh_*.tif")):
    tag = path.split('merged_coh_')[-1].replace('.tif','')
    try:
        ref_str, sec_str = tag.split('-')
        ref_date = pd.to_datetime(ref_str, format='%Y%m%d')
        sec_date = pd.to_datetime(sec_str, format='%Y%m%d')
        mid_date = ref_date + (sec_date - ref_date) / 2
    except Exception:
        continue
    records.append({"path": path, "mid_date": mid_date})

df_paths = pd.DataFrame(records)
if df_paths.empty:
    print('No merged coherence files found for calendar')
    raise SystemExit

# Apply current date range using midpoint date
date_start_day = dateStart.date()
date_end_day = dateEnd.date()
df_paths = df_paths[(df_paths['mid_date'].dt.date >= date_start_day) & (df_paths['mid_date'].dt.date <= date_end_day)]

# Calendar year labeling
if USE_WATER_YEAR:
    # Water year starts Oct (10) and ends Sep (9)
    df_paths['year'] = df_paths['mid_date'].dt.year + (df_paths['mid_date'].dt.month >= 10).astype(int)
    month_order = [10,11,12,1,2,3,4,5,6,7,8,9]
    month_labels = ['Oct','Nov','Dec','Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep']
else:
    df_paths['year'] = df_paths['mid_date'].dt.year
    month_order = list(range(1,13))
    month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

years = sorted(df_paths['year'].unique())

# Contrast stretch for low coherence (red = low)
# norm = mcolors.PowerNorm(gamma=0.3, vmin=0, vmax=1)

# One row per year, 12 columns
fig, axes = plt.subplots(len(years), 12, figsize=(24, 2.5*len(years)), constrained_layout=True)
if len(years) == 1:
    axes = np.array([axes])

for row_idx, y in enumerate(years):
    # pick a template for consistent grid within the year (first available file)
    year_paths = df_paths[df_paths['year'] == y]['path'].tolist()
    if not year_paths:
        continue
    template = rioxarray.open_rasterio(year_paths[0])[0]

    for col_idx, m in enumerate(month_order):
        ax = axes[row_idx, col_idx]
        month_paths = df_paths[(df_paths['year'] == y) & (df_paths['mid_date'].dt.month == m)]['path'].tolist()
        if USE_WATER_YEAR:
            year_for_month = y - 1 if m in (10, 11, 12) else y
        else:
            year_for_month = y
        title = f"{month_labels[col_idx]} {year_for_month}"
        if not month_paths:
            ax.set_title(title, fontsize=9)
            ax.set_xticks([])
            ax.set_yticks([])
            # keep a visible box for empty months
            for spine in ax.spines.values():
                spine.set_visible(True)
                spine.set_linewidth(0.8)
                spine.set_color('0.5')
            continue
        stacks = []
        for p in month_paths:
            da = rioxarray.open_rasterio(p)[0]
            da = da.rio.reproject_match(template)
            stacks.append(da)
        da_month = xr.concat(stacks, dim='stack').mean(dim='stack', skipna=True)
        minlon, minlat, maxlon, maxlat = da_month.rio.bounds()
        bbox = [minlon, maxlon, minlat, maxlat]
        im = ax.imshow(da_month.values, cmap='gray', vmin=0, vmax=1, origin='upper', extent=bbox, interpolation='none')
        ax.set_title(title, fontsize=9)
        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_linewidth(0.8)
            spine.set_color('0.5')
    # left-side year label
    if USE_WATER_YEAR:
        label = f"WY {y}"
    else:
        label = str(y)
    axes[row_idx, 0].set_ylabel(label, rotation=90, labelpad=6, fontsize=9)
    axes[row_idx, 0].yaxis.set_label_coords(-0.06, 0.5)

fig.colorbar(im, ax=axes, orientation='vertical', fraction=0.02, pad=0.02, label='Mean coherence')


In [ ]:
# Debug: list midpoint dates and their counts
df_paths[['mid_date']].sort_values('mid_date')
df_paths['mid_date'].dt.to_period('M').value_counts().sort_index()


## 11. Create GIF animations (Amplitude + Coherence)


In [ ]:
import glob
import os
import numpy as np
import pandas as pd
import imageio.v2 as imageio
import matplotlib.pyplot as plt
import rioxarray
import xarray as xr

# Output folders
gif_dir = f"{savedir}/gifs"
os.makedirs(gif_dir, exist_ok=True)

def _global_bounds(paths):
    bounds = []
    for p in paths:
        da = rioxarray.open_rasterio(p)[0]
        minx, miny, maxx, maxy = da.rio.bounds()
        bounds.append((minx, miny, maxx, maxy))
    minx = min(b[0] for b in bounds)
    miny = min(b[1] for b in bounds)
    maxx = max(b[2] for b in bounds)
    maxy = max(b[3] for b in bounds)
    return [minx, maxx, miny, maxy]

def _render_frames(tif_paths, out_dir, cmap, vmin=None, vmax=None, title_prefix="", extent=None, cbar_label=None, cbar_ticks=None):
    os.makedirs(out_dir, exist_ok=True)
    frames = []
    for p in tif_paths:
        da = rioxarray.open_rasterio(p)[0]
        if extent is None:
            minlon, minlat, maxlon, maxlat = da.rio.bounds()
            extent = [minlon, maxlon, minlat, maxlat]
        fig, ax = plt.subplots(figsize=(6,4))
        im = ax.imshow(da.values, cmap=cmap, origin='upper', extent=extent, vmin=vmin, vmax=vmax)
        tag = os.path.basename(p).replace('.tif','')
        ax.set_title(f"{title_prefix}{tag}", fontsize=9)
        ax.set_xticks([])
        ax.set_yticks([])
        cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02)
        if cbar_label:
            cb.set_label(cbar_label)
        if cbar_ticks is not None:
            cb.set_ticks(cbar_ticks)
        frame_path = os.path.join(out_dir, f"{tag}.png")
        fig.savefig(frame_path, dpi=150)
        plt.close(fig)
        frames.append(frame_path)
    return frames

def _pad_frames(frame_paths):
    imgs = [imageio.imread(f) for f in frame_paths]
    max_h = max(im.shape[0] for im in imgs)
    max_w = max(im.shape[1] for im in imgs)
    padded = []
    for im in imgs:
        pad_h = max_h - im.shape[0]
        pad_w = max_w - im.shape[1]
        padded.append(np.pad(im, ((0, pad_h), (0, pad_w), (0, 0)), mode='edge'))
    return padded

# Amplitude GIF (uses merged amplitude mosaics)
amp_paths = sorted(glob.glob(f"{savedir}/tifs/merged_amp_*.tif"))
amp_paths = [p for p in amp_paths if 'WGS84' not in p]
if amp_paths:
    _vals = []
    for p in amp_paths:
        da = rioxarray.open_rasterio(p)[0]
        _vals.append(da.values.ravel())
    _vals = np.concatenate(_vals)
    amp_vmin = np.nanpercentile(_vals, 1)
    amp_vmax = np.nanpercentile(_vals, 99)
    amp_extent = _global_bounds(amp_paths)
    amp_frames = _render_frames(
        amp_paths, f"{gif_dir}/amp_frames", cmap="gray", vmin=amp_vmin, vmax=amp_vmax,
        title_prefix="AMP_", extent=amp_extent, cbar_label="Amplitude (dB)"
    )
    amp_gif = f"{gif_dir}/amplitude.gif"
    amp_imgs = _pad_frames(amp_frames)
    imageio.mimsave(amp_gif, amp_imgs, duration=0.8)
    print(f"Wrote {amp_gif}")
else:
    print('No merged amplitude files found for GIF')

# Coherence GIF (uses merged coherence mosaics)
coh_paths = sorted(glob.glob(f"{savedir}/tifs/merged_coh_*.tif"))
coh_paths = [p for p in coh_paths if 'WGS84' not in p]
if coh_paths:
    coh_extent = _global_bounds(coh_paths)
    coh_frames = _render_frames(
        coh_paths, f"{gif_dir}/coh_frames", cmap='gray', vmin=0, vmax=1,
        title_prefix='COH_', extent=coh_extent, cbar_label='Coherence', cbar_ticks=[0,0.5,1]
    )
    coh_gif = f"{gif_dir}/coherence.gif"
    coh_imgs = _pad_frames(coh_frames)
    imageio.mimsave(coh_gif, coh_imgs, duration=0.8)
    print(f"Wrote {coh_gif}")
else:
    print('No merged coherence files found for GIF')


# Monthly mean coherence GIF (same monthly averaging as calendar)
if coh_paths:
    records = []
    for path in sorted(glob.glob(f"{savedir}/tifs/merged_coh_*.tif")):
        tag = path.split('merged_coh_')[-1].replace('.tif','')
        try:
            ref_str, sec_str = tag.split('-')
            ref_date = pd.to_datetime(ref_str, format='%Y%m%d')
            sec_date = pd.to_datetime(sec_str, format='%Y%m%d')
            mid_date = ref_date + (sec_date - ref_date) / 2
        except Exception:
            continue
        records.append({"path": path, "mid_date": mid_date})

    df_paths = pd.DataFrame(records)
    if df_paths.empty:
        print('No merged coherence files found for monthly GIF')
    else:
        # Apply current date range using midpoint date
        date_start_day = dateStart.date()
        date_end_day = dateEnd.date()
        df_paths = df_paths[(df_paths['mid_date'].dt.date >= date_start_day) & (df_paths['mid_date'].dt.date <= date_end_day)]

        # Month/year label logic
        if USE_WATER_YEAR:
            df_paths['year'] = df_paths['mid_date'].dt.year + (df_paths['mid_date'].dt.month >= 10).astype(int)
            month_order = [10,11,12,1,2,3,4,5,6,7,8,9]
            month_labels = ['Oct','Nov','Dec','Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep']
        else:
            df_paths['year'] = df_paths['mid_date'].dt.year
            month_order = list(range(1,13))
            month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

        # Frame output
        monthly_dir = f"{gif_dir}/coh_monthly_frames"
        os.makedirs(monthly_dir, exist_ok=True)
        monthly_frames = []

        years = sorted(df_paths['year'].unique())
        coh_extent = _global_bounds(coh_paths)

        for y in years:
            year_paths = df_paths[df_paths['year'] == y]['path'].tolist()
            if not year_paths:
                continue
            template = rioxarray.open_rasterio(year_paths[0])[0]

            for col_idx, m in enumerate(month_order):
                month_paths = df_paths[(df_paths['year'] == y) & (df_paths['mid_date'].dt.month == m)]['path'].tolist()
                if not month_paths:
                    continue

                stacks = []
                for p in month_paths:
                    da = rioxarray.open_rasterio(p)[0]
                    da = da.rio.reproject_match(template)
                    stacks.append(da)
                da_month = xr.concat(stacks, dim='stack').mean(dim='stack', skipna=True)

                if USE_WATER_YEAR:
                    year_for_month = y - 1 if m in (10, 11, 12) else y
                else:
                    year_for_month = y
                title = f"{month_labels[col_idx]} {year_for_month}"

                fig, ax = plt.subplots(figsize=(6,4))
                im = ax.imshow(da_month.values, cmap='gray', vmin=0, vmax=1, origin='upper', extent=coh_extent, interpolation='none')
                ax.set_title(title, fontsize=9)
                ax.set_xticks([])
                ax.set_yticks([])
                cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02)
                cb.set_label('Mean coherence')
                cb.set_ticks([0, 0.5, 1])
                tag = f"{year_for_month}_{m:02d}"
                frame_path = os.path.join(monthly_dir, f"{tag}.png")
                fig.savefig(frame_path, dpi=150)
                plt.close(fig)
                monthly_frames.append(frame_path)

        if monthly_frames:
            monthly_imgs = _pad_frames(monthly_frames)
            monthly_gif = f"{gif_dir}/coherence_monthly.gif"
            imageio.mimsave(monthly_gif, monthly_imgs, duration=0.8)
            print(f"Wrote {monthly_gif}")
        else:
            print('No monthly coherence frames created (no data in range)')
